# 09 Modern Transformer Activations (PyTorch SwiGLU & Autograd Derivatives)

Implementing SwiGLU FFN in PyTorch and evaluating PyTorch autograd derivatives for ReLU, GELU, and SiLU/SwiGLU.

## Part 1: PyTorch SwiGLU Feed-Forward Network Module

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.style.use('dark_background')

class PyTorchSwiGLUFFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w = nn.Linear(d_model, d_ff, bias=False)
        self.v = nn.Linear(d_model, d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.out_proj(F.silu(self.w(x)) * self.v(x))

swiglu = PyTorchSwiGLUFFN(d_model=256, d_ff=512)
x = torch.randn(2, 10, 256)
out = swiglu(x)
print('PyTorch SwiGLU FFN Output Shape:', out.shape)

## Part 2: PyTorch Autograd Activation Curves & Derivatives (ReLU vs GELU vs SiLU)
Tracking autograd gradients $\frac{d f(x)}{dx}$ across activation functions.

In [ ]:
x = torch.linspace(-4, 4, 400, requires_grad=True)

F.relu(x).sum().backward()
relu_g = x.grad.clone(); x.grad.zero_()

F.gelu(x).sum().backward()
gelu_g = x.grad.clone(); x.grad.zero_()

F.silu(x).sum().backward()
silu_g = x.grad.clone()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(x.detach().numpy(), F.relu(x.detach()).numpy(), label='nn.ReLU()', color='#ff6b6b')
ax1.plot(x.detach().numpy(), F.gelu(x.detach()).numpy(), label='nn.GELU()', color='#4dabf7')
ax1.plot(x.detach().numpy(), F.silu(x.detach()).numpy(), label='nn.SiLU() (SwiGLU)', color='#51cf66')
ax1.set_title('PyTorch Activations f(x)'); ax1.grid(True); ax1.legend()

ax2.plot(x.detach().numpy(), relu_g.numpy(), label='nn.ReLU Gradient (Hard 0 cutoff)', color='#ff6b6b')
ax2.plot(x.detach().numpy(), gelu_g.numpy(), label='nn.GELU Autograd Derivative', color='#4dabf7')
ax2.plot(x.detach().numpy(), silu_g.numpy(), label='nn.SiLU Autograd Derivative', color='#51cf66')
ax2.set_title('PyTorch Autograd Derivatives f\'(x)'); ax2.grid(True); ax2.legend()
plt.tight_layout()
plt.show()